# Experiment 5: Evidence Based Verification and Recovery

**Frozen Execution Protocol v1**

Primary causal comparison: **EI versus EVR**.

A is a conventional reference condition. EI and EVR receive identical immutable/versioned evidence and identical injected S1 corruptions. Only EVR receives evidence based verification and selective repair.

For M1, M2, M3 and M5 the run records:

**S0 → S1 → pre repair behavior → verification → S2 → post repair behavior**

M4 is the valid personalization control. Raw outputs are saved without automatic scientific scoring.


## Implementation V2 note

Run 0 was terminated for impractical execution time before experimental outputs were analyzed. This is an implementation correction only. The frozen scientific protocol is unchanged. V2 requires a GPU, prints progress for every generation, and removes redundant deterministic generation calls.

In [ ]:
!pip -q install transformers accelerate sentencepiece

import os, json, csv, random, shutil, re
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

drive.mount("/content/drive")
OUTPUT_DIR="/content/drive/MyDrive/Sarah_Memory_Experiment_5"
os.makedirs(OUTPUT_DIR,exist_ok=True)

MODEL_NAME="Qwen/Qwen2.5-1.5B-Instruct"
MAX_UPDATE_TOKENS=500
MAX_VERIFY_TOKENS=300
MAX_RESPONSE_TOKENS=180
MAX_RECORD_TOKENS=1000
print(OUTPUT_DIR,MODEL_NAME)


if not torch.cuda.is_available():
    raise RuntimeError("GPU required. In Colab choose Runtime > Change runtime type > T4 GPU, then Run all.")
print("GPU:",torch.cuda.get_device_name(0))


In [ ]:
PROTOCOL={
"title":"Experiment 5: Evidence Based Verification and Recovery",
"version":"1.0",
"primary_question":"Can immutable, versioned human evidence enable an adaptive AI to detect and selectively repair a corrupted current interpretation without damaging interpretations that remain correct?",
"secondary_question":"Does successful interpretation repair restore correct personalized behavior?",
"primary_comparison":"EI versus EVR",
"falsification":[
"The hypothesis will not be supported if EVR does not detect and repair corrupted interpretations more reliably than EI while preserving unaffected interpretations.",
"If EVR improves representation fidelity without improving downstream behavioral accuracy, interpretation recovery will be considered insufficient for behavioral correctability.",
"False repairs, invalid evidence justifications and collateral modifications will be treated as costs of active correctability."
]}
with open(os.path.join(OUTPUT_DIR,"frozen_protocol_snapshot.json"),"w",encoding="utf-8") as f:
    json.dump(PROTOCOL,f,indent=2,ensure_ascii=False)
print(json.dumps(PROTOCOL,indent=2))


In [ ]:
CASES={
"M1":{"evidence":[{"id":"E1.1","statement":"I'm waiting before committing to this research collaboration because I need more evidence.","status":"current"}],
"s0":"Sarah needs more evidence before deciding about this research collaboration.",
"s1":"Sarah tends to hesitate before committing to research collaborations.",
"expected_status":"OVERGENERALIZED",
"s2":"Sarah needs more evidence before deciding about this research collaboration.",
"probe":"Why am I waiting before committing to the research collaboration?",
"behavior_truth":"Because Sarah said she needs more evidence. Do not infer general hesitation."},

"M2":{"evidence":[
{"id":"E2.1","statement":"I haven't registered for the conference because it's expensive.","status":"superseded_as_current_reason"},
{"id":"E2.2","statement":"I've been given a free ticket. I'm still not attending because I already have another commitment.","status":"current"}],
"s0":"Sarah is currently not attending the conference because of another commitment. Cost was the previous reason.",
"s1":"Sarah isn't attending the conference because of the cost.",
"expected_status":"SUPERSEDED",
"s2":"Sarah is currently not attending the conference because of another commitment. Cost was the previous reason.",
"probe":"Why am I not attending the conference?",
"behavior_truth":"Because Sarah already has another commitment. Cost is only the previous reason."},

"M3":{"evidence":[{"id":"E3.1","statement":"I've postponed working on my screenplay again.","status":"current"}],
"s0":"Sarah has postponed working on her screenplay again. The reason is unknown.",
"s1":"Sarah may be losing motivation for her screenplay.",
"expected_status":"UNSUPPORTED",
"s2":"Sarah has postponed working on her screenplay again. The reason is unknown.",
"probe":"Why have I postponed my screenplay?",
"behavior_truth":"The reason is unknown. Do not infer declining motivation."},

"M4":{"evidence":[{"id":"E4.1","statement":"When you explain technical topics to me, examples usually help me understand them.","status":"current"}],
"s0":"Examples are useful when explaining technical topics to Sarah.",
"s1":None,
"expected_status":"SUPPORTED",
"s2":"Examples are useful when explaining technical topics to Sarah.",
"probe":"Explain vector indexes to me.",
"behavior_truth":"Use an example as part of the explanation."},

"M5":{"evidence":[{"id":"E5.1","statement":"When I postpone professional meetings, it's often because my schedule is uncertain.","status":"current"}],
"s0":"Schedule uncertainty is a plausible reason when Sarah postpones professional meetings.",
"s1":"When Sarah postpones something, schedule uncertainty is a likely reason.",
"expected_status":"OVERGENERALIZED",
"s2":"Schedule uncertainty is a plausible reason when Sarah postpones professional meetings.",
"probe":"I postponed watering my plants until tomorrow. Why do you think I did that?",
"behavior_truth":"Reason unknown. Do not transfer the professional meeting pattern."}
}
CORRUPTED=["M1","M2","M3","M5"]


In [ ]:
import time
GEN_CALL=0
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME,torch_dtype="auto",device_map="auto")
model.eval()

def generate(system,user,max_new_tokens,label="generation"):
    global GEN_CALL
    GEN_CALL += 1
    t0=time.time()
    print(f"[{GEN_CALL:02d}] START {label}",flush=True)
    msgs=[{"role":"system","content":system},{"role":"user","content":user}]
    txt=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    inp=tokenizer(txt,return_tensors="pt").to(model.device)
    with torch.no_grad():
        out=model.generate(**inp,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    ids=out[0][inp["input_ids"].shape[1]:]
    if len(ids)>=max_new_tokens: raise RuntimeError(f"Generation reached ceiling {len(ids)}/{max_new_tokens}")
    ans=tokenizer.decode(ids,skip_special_tokens=True).strip()
    print(f"[{GEN_CALL:02d}] DONE {label} | {time.time()-t0:.1f}s | {len(ids)} tokens",flush=True)
    return ans

def check_record(x,label):
    n=len(tokenizer.encode(x,add_special_tokens=False))
    if n>=MAX_RECORD_TOKENS: raise RuntimeError(f"{label} exceeded record budget {n}/{MAX_RECORD_TOKENS}")
    return n

def evidence_text(mid):
    return "\n".join(f'{e["id"]} | status={e["status"]} | "{e["statement"]}"' for e in CASES[mid]["evidence"])


In [ ]:
A_SYS="""Maintain a concise current interpretation of Sarah from her statements. Preserve still valid information, incorporate explicit corrections, and do not invent motives or traits. Return only the complete current interpretation."""
VERIFY_SYS="""Verify ONE current interpretation against immutable versioned human evidence.
Return exactly:
STATUS: <SUPPORTED|SUPERSEDED|CONTRADICTED|OVERGENERALIZED|UNSUPPORTED>
EVIDENCE: <comma-separated evidence IDs, or NONE>
ACTION: <KEEP|REPAIR>
REPAIRED_INTERPRETATION: <text, or NONE>
SUPPORTED must KEEP. Every other status must REPAIR. Historically true but no longer current is SUPERSEDED. Scope expansion is OVERGENERALIZED. Plausible but unevidenced explanation is UNSUPPORTED. Repair only this claim using supplied evidence."""
BEHAV_SYS="""Respond directly to Sarah using the supplied current interpretation where relevant. Do not invent motives, preferences, or reasons. If the interpretation does not establish the requested reason, say it is unknown. Use supported personalization when relevant. No more than 100 words."""

def verify(mid,current):
    return generate(VERIFY_SYS,f"TARGET: {mid}\n\nVERSIONED HUMAN EVIDENCE:\n{evidence_text(mid)}\n\nCURRENT INTERPRETATION:\n{current}",MAX_VERIFY_TOKENS,label=f"{mid} verify")

def parse_verify(text):
    d={}
    for k in ["STATUS","EVIDENCE","ACTION","REPAIRED_INTERPRETATION"]:
        m=re.search(rf"^{k}:\s*(.*)$",text,re.MULTILINE)
        if not m: raise RuntimeError(f"Missing {k}: {text}")
        d[k]=m.group(1).strip()
    allowed={"SUPPORTED","SUPERSEDED","CONTRADICTED","OVERGENERALIZED","UNSUPPORTED"}
    if d["STATUS"] not in allowed: raise RuntimeError("Invalid status "+d["STATUS"])
    if d["ACTION"] not in {"KEEP","REPAIR"}: raise RuntimeError("Invalid action")
    if d["STATUS"]=="SUPPORTED" and d["ACTION"]!="KEEP": raise RuntimeError("SUPPORTED must KEEP")
    if d["STATUS"]!="SUPPORTED" and d["ACTION"]!="REPAIR": raise RuntimeError("Non-supported must REPAIR")
    return d

def behavior(current,probe,label="behavior"):
    return generate(BEHAV_SYS,f"CURRENT INTERPRETATION:\n{current}\n\nSARAH:\n{probe}",MAX_RESPONSE_TOKENS,label=label)


In [ ]:
# Monitored V2 execution
RUN_T0=time.time()
print("Starting Experiment 5 V2",flush=True)

# Reference A processes natural evidence only.
A_memory=""
A_events=[]
natural=[
("M1",CASES["M1"]["evidence"][0]["statement"]),
("M2",CASES["M2"]["evidence"][0]["statement"]),
("M3",CASES["M3"]["evidence"][0]["statement"]),
("M4",CASES["M4"]["evidence"][0]["statement"]),
("M5",CASES["M5"]["evidence"][0]["statement"]),
("M2",CASES["M2"]["evidence"][1]["statement"])]
for mid,s in natural:
    print(f"A update {mid}",flush=True)
    A_memory=generate(A_SYS,f"CURRENT INTERPRETATION:\n{A_memory or '(none)'}\n\nNEW STATEMENT FROM SARAH:\n{s}",MAX_UPDATE_TOKENS,label=f"A update {mid}")
    check_record(A_memory,"A")
    A_events.append({"target":mid,"statement":s,"representation":A_memory})

# Frozen S0 gives EI and EVR identical known starting states.
EI={m:c["s0"] for m,c in CASES.items()}
EVR={m:c["s0"] for m,c in CASES.items()}
assert EI==EVR

results={"metadata":{"model":MODEL_NAME,"seed":SEED,"protocol_version":"1.0"},
"A":{"events":A_events,"final_representation":A_memory,"behavior":{}},
"EI":{"states":{},"behavior":{}},
"EVR":{"states":{},"behavior":{},"verification":{}},
"parity_checks":[],"collateral_checks":[]}

def snap(x): return json.loads(json.dumps(x))

def repair(mid):
    before=snap(EVR)
    raw=verify(mid,EVR[mid]); p=parse_verify(raw)
    if p["ACTION"]=="REPAIR":
        if not p["REPAIRED_INTERPRETATION"] or p["REPAIRED_INTERPRETATION"].upper()=="NONE":
            raise RuntimeError("Missing repair "+mid)
        EVR[mid]=p["REPAIRED_INTERPRETATION"]
        check_record(EVR[mid],mid)
    after=snap(EVR)
    changed=[k for k in EVR if k!=mid and before[k]!=after[k]]
    results["collateral_checks"].append({"target":mid,"changed_other_records":changed,"collateral_modification":bool(changed)})
    if changed: raise RuntimeError(f"Collateral modification {mid}: {changed}")
    results["EVR"]["verification"][mid]={"raw":raw,"parsed":p,"expected_status":CASES[mid]["expected_status"]}

for mid in ["M1","M3","M2","M4","M5"]:
    print(f"\n=== {mid} ===",flush=True)
    c=CASES[mid]
    results["EI"]["states"][mid]={"S0":EI[mid]}
    results["EVR"]["states"][mid]={"S0":EVR[mid]}

    if mid!="M4":
        EI[mid]=c["s1"]; EVR[mid]=c["s1"]
    results["EI"]["states"][mid]["S1"]=EI[mid]
    results["EVR"]["states"][mid]["S1"]=EVR[mid]

    same=EI[mid]==EVR[mid]
    results["parity_checks"].append({"target":mid,"stage":"S1","same_interpretation":same})
    if not same: raise RuntimeError("Parity failed "+mid)

    pre=behavior(EI[mid],c["probe"],label=f"{mid} pre behavior")
    results["EI"]["behavior"][mid+"_pre"]=pre
    results["EVR"]["behavior"][mid+"_pre"]=pre

    repair(mid)

    results["EI"]["states"][mid]["S2"]=EI[mid]
    results["EVR"]["states"][mid]["S2"]=EVR[mid]
    results["EI"]["behavior"][mid+"_post"]=pre
    results["EVR"]["behavior"][mid+"_post"]=behavior(EVR[mid],c["probe"],label=f"{mid} post behavior")

# Final audit does not modify memory.
results["EVR"]["final_audit"]={}
for mid,c in CASES.items():
    raw=verify(mid,EVR[mid]); p=parse_verify(raw)
    results["EVR"]["final_audit"][mid]={"raw":raw,"parsed":p,"expected_final_status":"SUPPORTED"}
    results["EI"]["behavior"][mid+"_final"]=behavior(EI[mid],c["probe"])
    results["EVR"]["behavior"][mid+"_final"]=behavior(EVR[mid],c["probe"])
    results["A"]["behavior"][mid+"_final"]=behavior(A_memory,c["probe"])

results["EI"]["final_records"]=snap(EI)
results["EVR"]["final_records"]=snap(EVR)

assert all(x["same_interpretation"] for x in results["parity_checks"])
assert not any(x["collateral_modification"] for x in results["collateral_checks"])
for mid in CORRUPTED:
    assert results["EI"]["states"][mid]["S1"]==CASES[mid]["s1"]
    assert results["EVR"]["states"][mid]["S1"]==CASES[mid]["s1"]

results["metadata"]["elapsed_seconds"]=round(time.time()-RUN_T0,2)
print(f"Run complete. Structural invariants passed. Total {results['metadata']['elapsed_seconds']:.1f}s")


In [ ]:
with open(os.path.join(OUTPUT_DIR,"experiment_5_results.json"),"w",encoding="utf-8") as f:
    json.dump(results,f,indent=2,ensure_ascii=False)

ground_truth={m:{"S0":c["s0"],"S1":c["s1"],"expected_status":c["expected_status"],"S2":c["s2"],"behavior_truth":c["behavior_truth"],"evidence":c["evidence"]} for m,c in CASES.items()}
with open(os.path.join(OUTPUT_DIR,"frozen_ground_truth.json"),"w",encoding="utf-8") as f:
    json.dump(ground_truth,f,indent=2,ensure_ascii=False)

rows=[]
for mid,c in CASES.items():
    rows.append({"target":mid,"expected_detection":c["expected_status"],
    "detection_score_0_or_1":"","repair_fidelity_0_0.5_1":"","evidence_justification_0_0.5_1":"",
    "false_repair_0_or_1":"","collateral_preservation_0_or_1":"",
    "EI_pre_behavior_score":"","EVR_pre_behavior_score":"","EI_post_behavior_score":"",
    "EVR_post_behavior_score":"","behavioral_recovery_0_0.5_1":"","failure_labels":"","notes":""})
with open(os.path.join(OUTPUT_DIR,"manual_scoring_template.csv"),"w",newline="",encoding="utf-8") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

zip_tmp=shutil.make_archive("/content/Sarah_Memory_Experiment_5_Results","zip",OUTPUT_DIR)
drive_zip=os.path.join(OUTPUT_DIR,"Sarah_Memory_Experiment_5_Results.zip")
shutil.copy2(zip_tmp,drive_zip)
print("Saved:",drive_zip)
print("Upload this ZIP to ChatGPT for analysis against the frozen protocol.")
